# A simple Backtracking Constraint Solver

We use the class `RecursiveSet`.  Furthermore, we load the file `ConstraintCompiler.jsh`, which is discussed below.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load ConstraintCompiler.jsh

A `CSP`(short for <em><u>C</u>onstraint <u>S</u>atisfaction <u>P</u>roblem</em>) is defined as a tripel of the form
$$ \langle \texttt{Vars}, \texttt{Values}, \texttt{Constraints} \rangle $$
where 
 * $\texttt{Vars}$ is a list of *variables*,
 * $\texttt{Values}$ is a list of *values* that these variables can take, and 
 * $\texttt{Constraints}$ is a list of formulas describing *conditions* 
   that the variables must satisfy. 
   
   These formulas are given as strings.  These strings have to be valid
   Java expressions that evaluate to a Boolean value.
   
To simplify the implementation we assume that the set $\texttt{Values}$ is a set of integers.  A CSP is represented as an object of the following record class.

In [ ]:
record CSP(List<String> variables, List<Integer> values, List<String> constraints) {}

An `Assignment` maps **some** variables to values.  It is represented as an object of type `Map<String, Integer>`.

An `AnnotatedConstraint` is a triple consisting of a `formula`, the set `variables` of all variables occurring in this formula, and a `predicate` that evaluates the formula for a given assignment.

In [ ]:
// A constraint together with the set of its variables and a compiled predicate that
// evaluates the constraint for a given variable assignment.
record AnnotatedConstraint(String formula,
                           RecursiveSet<String> variables,
                           Predicate<Map<String, Integer>> predicate) {}

In [ ]:
@SafeVarargs
<T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions, i.e. the names `Math`, `true`, and `false` are not treated as variables.  Furthermore, a name that follows a dot, like `abs` in `Math.abs`, is not a variable.

As I am working here with *regular expressions*, **you do not need to understand what is going on**.

In [ ]:
// Returns the set of variables occurring in the expression expr.  Identifiers that
// follow a dot (like abs in Math.abs) and the names Math, true, and false are not
// variables.
RecursiveSet<String> collectVariables(String expr) {
    Pattern identifier = Pattern.compile("(?<!\\.)\\b[a-zA-Z_][a-zA-Z0-9_]*\\b");
    Matcher matcher    = identifier.matcher(expr);
    RecursiveSet<String> variables = new RecursiveSet<>();
    while (matcher.find()) {
        String candidate = matcher.group();
        if (!Set.of("Math", "true", "false").contains(candidate)) {
            variables.add(candidate);
        }
    }
    return variables;
}

In [ ]:
var expr = "Math.abs(x - y) + Math.abs(z1 - z2) == 0";
collectVariables(expr)

In order to evaluate a formula like `x * y == z + 1` for a given assignment, JavaScript and Python provide a function called `eval`, which takes a string and evaluates it as an expression.  Java does not provide such a function.  However, the JDK contains the Java compiler and this compiler can be invoked from a running program.  The file `ConstraintCompiler.jsh` defines the function
```java
List<Predicate<Map<String, Integer>>> compileConstraints(List<String> exprs, List<RecursiveSet<String>> variablesList)
```
that takes a list of Boolean expressions together with the sets of the variables occurring in these expressions.  It returns a list of *predicates*: The predicate `predicates.get(i)` takes an assignment and evaluates the expression `exprs.get(i)` using this assignment.  As this function is quite technical, **you do not need to understand its implementation**.

In [ ]:
var predicates = compileConstraints(List.of("x * y == z + 1"), List.of(set("x", "y", "z")));
predicates.get(0).test(Map.of("x", 2, "y", 3, "z", 5))

The function `isConsistent` takes four arguments:
- `variable` is a variable.
- `value` is a value that is to be assigned to the variable `variable`.
- `assignment` is a partial variable assignment that does not assign a value for `variable`
  and that is *consistent* with all constraints in the set `constraints`.
- `assignedVars` is the set of variables that are assigned a value by `assignment`.
- `constraints` is a list of annotated logical formulas.

The function checks whether the assignment
$$ \texttt{assignment} \cup \{\texttt{variable} \mapsto \texttt{value}\}$$
violates any of the formulas in `constraints`. It assumes that
`assignment` is *consistent*, i.e. all constraints that can already be evaluated with `assignment`
are evaluated as `true`.

In [ ]:
// Checks whether the assignment assignment ∪ {variable ↦ value} is consistent with
// all constraints that can be evaluated using this assignment.
boolean isConsistent(String variable, int value,
                     Map<String, Integer> assignment,
                     RecursiveSet<String> assignedVars,
                     List<AnnotatedConstraint> constraints) {
    var newAssignment = new HashMap<>(assignment);
    newAssignment.put(variable, value);
    var newAssignedVars = assignedVars.union(set(variable));
    return constraints.stream().allMatch(c -> {
        boolean canEvaluate = c.variables().contains(variable) && c.variables().isSubset(newAssignedVars);
        return !canEvaluate || c.predicate().test(newAssignment);
    });
}

Given a **consistent** *partial variable assignment* `assignment` and a constraint satisfaction problem `csp`,
the function `backtrackSearch(assignment, csp)` tries to extend the given assignment recursively and thereby produce a solution of the given CSP.

If the given assignment can not be extended to a solution, the function returns `null`.

In [ ]:
// Tries to extend the consistent partial assignment to a solution of the CSP.
Map<String, Integer> backtrackSearch(Map<String, Integer> assignment,
                                     RecursiveSet<String> assignedVars,
                                     List<String> variables,
                                     List<Integer> values,
                                     List<AnnotatedConstraint> constraints) {
    if (assignment.size() == variables.size()) {
        return assignment;
    }
    String nextVar = variables.stream().filter(v -> !assignedVars.contains(v)).findFirst().get();
    for (int value : values) {
        if (isConsistent(nextVar, value, assignment, assignedVars, constraints)) {
            var newAssignment = new LinkedHashMap<>(assignment);
            newAssignment.put(nextVar, value);
            var newAssignedVars = assignedVars.union(set(nextVar));
            var result = backtrackSearch(newAssignment, newAssignedVars, variables, values, constraints);
            if (result != null) {
                return result;
            }
        }
    }
    return null;
}

The input to the function `solve` is a *constraint satisfaction problem*, i.e. a *CSP*.
The function `solve` tries to compute a solution of this problem via *backtracking*.
Its main purpose is to transform the given *CSP* into an *annotated CSP* where all the formulas
are *annotated* with their variables. That is, the third component of the *CSP* is now no longer a list of formulas but rather a list of triples of the form `(f, V, P)` where `f` is a formula, `V` is the set of variables occurring in this formula, and `P` is the predicate that results from compiling `f`. The function then calls the auxiliary function `backtrackSearch` that recursively solves the *annotated CSP*.

In [ ]:
// Solves the given CSP.  Returns null if the CSP is not solvable.
Map<String, Integer> solve(CSP csp) {
    var Variables   = csp.constraints().stream().map(f -> collectVariables(f)).toList();
    var predicates  = compileConstraints(csp.constraints(), Variables);
    List<AnnotatedConstraint> annotatedConstraints = new ArrayList<>();
    for (int i = 0; i < csp.constraints().size(); ++i) {
        annotatedConstraints.add(new AnnotatedConstraint(csp.constraints().get(i),
                                                         Variables.get(i),
                                                         predicates.get(i)));
    }
    return backtrackSearch(new LinkedHashMap<>(), set(), csp.variables(), csp.values(), annotatedConstraints);
}

In [ ]:
// Example usage to test the new function:
var myCSP = new CSP(
    List.of("x", "y", "z"),         // Variables
    List.of(1, 2, 3, 4, 5),         // Value Domain
    List.of(                        // Constraints
        "x * y == z + 1",
        "x + y == z",
        "x * y == 2 * z - x * x"
    )
);

In [ ]:
var solution = solve(myCSP);

if (solution != null) {
    System.out.println("Solution found!");
    System.out.println("x = " + solution.get("x"));
    System.out.println("y = " + solution.get("y"));
    System.out.println("z = " + solution.get("z"));
} else {
    System.out.println("No solution exists.");
}

The functions defined in this notebook are also available in the file `02-Backtracking-Constraint-Solver.jsh`.  This file is used by later notebooks.